# 00 — CONUS404 water-balance data preparation

The notebook prepares the CONUS404 precipitation, actual evapotranspiration (AET), and runoff inputs used by the paper analyses. The workflow reads the public HyTEST Open Storage Network (OSN) Zarr products, aggregates one period at a time, reprojects the results, and writes analysis-ready GeoTIFFs.

It consolidates the logic from the four exploratory notebooks supplied with the project:

- `CONUS404_Annual_Precipitation.ipynb`
- `CONUS404_Annual_AET_data.ipynb`
- `CONUS404_Seasonal_RAIN_data.ipynb`
- `CONUS404_Seasonal_RAIN_data-BA.ipynb`

The workflow accesses the cloud-native daily product and computes only the configured variables and periods. This direct temporal aggregation replaces the redundant monthly NetCDF intermediates in the exploratory precipitation notebooks. CONUS404 spans water years 1980–2022.

## Variable choices preserved from the source notebooks

| Product | Default variable/formula | Alternative | Note |
|---|---|---|---|
| Native precipitation | `PREC_ACC_NC` | `ACRAINLSM + ACSNOWLSM` | The workflow preserves `PREC_ACC_NC` from the annual source and the rain-plus-snow option from the seasonal source. |
| Bias-adjusted precipitation | `RAIN` | — | The workflow reads `RAIN` from the HyTEST bias-adjusted daily product. |
| AET | `ACEDIR + ACETRAN + ACECAN` | `ACETLSM` | The workflow preserves the component sum from the AET source and retains `ACETLSM` as a diagnostic alternative. |
| Runoff | `ACRUNSF + ACQRF` | — | The workflow sums accumulated surface runoff and accumulated subsurface runoff. |

The workflow treats AET as accumulated **net** evaporation and does not clip negative values because condensation or deposition can be physically meaningful. The workflow calculates total runoff as the sum of the two accumulated runoff components.

## Sources

- CONUS404 dataset and DOI: https://doi.org/10.5065/ZYY0-Y036
- NCAR GDEX dataset page: https://gdex.ucar.edu/datasets/d559000/
- HyTEST access guide: https://hytest-org.github.io/hytest/dataset_access/CONUS404_ACCESS.html
- HyTEST catalog: https://github.com/hytest-org/hytest/blob/main/dataset_catalog/subcatalogs/conus404-catalog.yml
- HyTEST changelog: https://hytest-org.github.io/hytest/dataset_access/CONUS404_CHANGELOG.html


## 1. Imports and configuration

`RUN_PREVIEW` and `RUN_EXPORTS` control cloud sampling and GeoTIFF generation.

Annual products are labeled by **water year** (October through September). Seasonal products preserve the supplied notebook's **calendar-year** labels: Fall = Oct–Dec, Winter = Jan–Mar, Spring = Apr–Jun, and Summer = Jul–Sep.


In [ ]:
from pathlib import Path
import warnings

import dask
import intake
import matplotlib.pyplot as plt
import metpy
import numpy as np
import pandas as pd
import rioxarray
import xarray as xr
from distributed import Client

START_WATER_YEAR = 1981
END_WATER_YEAR = 2020
START_SEASON_YEAR = 1981
END_SEASON_YEAR = 2020

EXPORT_ANNUAL = True
EXPORT_SEASONAL = True
EXPORT_NATIVE_PRECIPITATION = True
EXPORT_BIAS_ADJUSTED_PRECIPITATION = True
EXPORT_AET = True
EXPORT_RUNOFF = True

NATIVE_PRECIPITATION_METHOD = "PREC_ACC_NC"

AET_METHOD = "components"

TARGET_CRS = "EPSG:5070"
OUTPUT_NODATA = np.nan
OVERWRITE = False

RUN_PREVIEW = True
RUN_EXPORTS = False
START_LOCAL_DASK_CLIENT = True
DASK_WORKERS = 2
THREADS_PER_WORKER = 2
MEMORY_LIMIT_PER_WORKER = "4GB"

CATALOG_URL = (
    "https://raw.githubusercontent.com/hytest-org/hytest/"
    "main/dataset_catalog/hytest_intake_catalog.yml"
)
NATIVE_DATASET = "conus404-daily-osn"
BIAS_ADJUSTED_DATASET = "conus404-daily-ba-osn"

def find_repository_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config.example.yml").exists() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError(
        "Repository root not found: config.example.yml and src/ are unavailable."
    )

REPO_ROOT = find_repository_root(Path.cwd())
DATA_ROOT = REPO_ROOT / "data"

OUTPUT_DIRECTORIES = {
    "annual_native_precipitation": DATA_ROOT / "annual" / "conus404",
    "annual_bias_adjusted_precipitation": DATA_ROOT / "annual" / "conus404ba",
    "annual_aet": DATA_ROOT / "annual" / "aet_conus404",
    "annual_runoff": DATA_ROOT / "annual" / "runoff_conus404",
    "seasonal_native_precipitation": DATA_ROOT / "seasonal" / "conus404",
    "seasonal_bias_adjusted_precipitation": DATA_ROOT / "seasonal" / "conus404ba",
    "seasonal_aet": DATA_ROOT / "seasonal" / "aet_conus404",
}

if START_WATER_YEAR > END_WATER_YEAR:
    raise ValueError("START_WATER_YEAR must not be later than END_WATER_YEAR.")
if START_SEASON_YEAR > END_SEASON_YEAR:
    raise ValueError("START_SEASON_YEAR must not be later than END_SEASON_YEAR.")
if NATIVE_PRECIPITATION_METHOD not in {"PREC_ACC_NC", "land_surface_components"}:
    raise ValueError("Invalid NATIVE_PRECIPITATION_METHOD.")
if AET_METHOD not in {"components", "ACETLSM"}:
    raise ValueError("Invalid AET_METHOD.")

print("Repository:", REPO_ROOT)
print("GeoTIFF root:", DATA_ROOT)
print("Exports enabled:", RUN_EXPORTS)


## 2. Dask and HyTEST catalog

The workflow uses the public anonymous OSN entries because the HyTEST changelog reported that the former AWS S3 copy was archived to Glacier in April 2025.


In [ ]:
client = None
if START_LOCAL_DASK_CLIENT:
    client = Client(
        n_workers=DASK_WORKERS,
        threads_per_worker=THREADS_PER_WORKER,
        memory_limit=MEMORY_LIMIT_PER_WORKER,
    )
    display(client)

catalog = intake.open_catalog(CATALOG_URL)
conus404_catalog = catalog["conus404-catalog"]
print("Available CONUS404 entries:")
print(list(conus404_catalog))

native_ds = conus404_catalog[NATIVE_DATASET].to_dask().metpy.parse_cf()
bias_adjusted_ds = None
if EXPORT_BIAS_ADJUSTED_PRECIPITATION:
    bias_adjusted_ds = (
        conus404_catalog[BIAS_ADJUSTED_DATASET].to_dask().metpy.parse_cf()
    )

print("Native time coverage:", native_ds.time.min().values, "to", native_ds.time.max().values)
if bias_adjusted_ds is not None:
    print(
        "Bias-adjusted time coverage:",
        bias_adjusted_ds.time.min().values,
        "to",
        bias_adjusted_ds.time.max().values,
    )


## 3. Variable validation and metadata

The workflow validates every selected variable before computation and records its units, description, and integration interval. The workflow requires accumulated water-depth variables in millimeters. The HyTEST daily product is derived from the hourly CONUS404 product.


In [ ]:
def required_native_variables() -> set[str]:
    variables: set[str] = set()
    if EXPORT_NATIVE_PRECIPITATION:
        if NATIVE_PRECIPITATION_METHOD == "PREC_ACC_NC":
            variables.add("PREC_ACC_NC")
        else:
            variables.update(("ACRAINLSM", "ACSNOWLSM"))
    if EXPORT_AET:
        if AET_METHOD == "components":
            variables.update(("ACEDIR", "ACETRAN", "ACECAN"))
        else:
            variables.add("ACETLSM")
    if EXPORT_RUNOFF:
        variables.update(("ACRUNSF", "ACQRF"))
    return variables

def validate_variables(dataset: xr.Dataset, variables: set[str], product_name: str) -> None:
    missing = sorted(variables.difference(dataset.data_vars))
    if missing:
        raise KeyError(f"{product_name} is missing required variables: {missing}")

native_variables = required_native_variables()
validate_variables(native_ds, native_variables, "native CONUS404")

if EXPORT_BIAS_ADJUSTED_PRECIPITATION:
    validate_variables(bias_adjusted_ds, {"RAIN"}, "bias-adjusted CONUS404")

metadata_rows = []
for variable in sorted(native_variables):
    attrs = native_ds[variable].attrs
    metadata_rows.append(
        {
            "product": "native",
            "variable": variable,
            "units": attrs.get("units"),
            "description": attrs.get("description", attrs.get("long_name")),
            "integration_length": attrs.get("integration_length"),
        }
    )
if EXPORT_BIAS_ADJUSTED_PRECIPITATION:
    attrs = bias_adjusted_ds["RAIN"].attrs
    metadata_rows.append(
        {
            "product": "bias-adjusted",
            "variable": "RAIN",
            "units": attrs.get("units"),
            "description": attrs.get("description", attrs.get("long_name")),
            "integration_length": attrs.get("integration_length"),
        }
    )

metadata = pd.DataFrame(metadata_rows)
display(metadata)

unexpected_units = metadata.loc[
    ~metadata["units"].astype(str).str.lower().isin({"mm", "millimeter", "millimeters"})
]
if not unexpected_units.empty:
    warnings.warn(
        "One or more selected variables do not advertise millimeter units."
    )


## 4. Precipitation, AET, and runoff fields

The workflow defines each water-balance field explicitly and attaches a `construction` attribute so every output retains the variable or component formula used in its calculation.


In [ ]:
def native_precipitation(dataset: xr.Dataset) -> xr.DataArray:
    if NATIVE_PRECIPITATION_METHOD == "PREC_ACC_NC":
        result = dataset["PREC_ACC_NC"]
        construction = "PREC_ACC_NC"
    else:
        result = dataset["ACRAINLSM"] + dataset["ACSNOWLSM"]
        construction = "ACRAINLSM + ACSNOWLSM"
    result = result.rename("precipitation")
    result.attrs.update(
        units="mm",
        long_name="CONUS404 accumulated precipitation",
        construction=construction,
    )
    return result

def bias_adjusted_precipitation(dataset: xr.Dataset) -> xr.DataArray:
    result = dataset["RAIN"].rename("precipitation")
    result.attrs.update(
        units="mm",
        long_name="Bias-adjusted CONUS404 accumulated precipitation",
        construction="RAIN",
    )
    return result

def actual_evapotranspiration(dataset: xr.Dataset) -> xr.DataArray:
    if AET_METHOD == "components":
        result = dataset["ACEDIR"] + dataset["ACETRAN"] + dataset["ACECAN"]
        construction = "ACEDIR + ACETRAN + ACECAN"
    else:
        result = dataset["ACETLSM"]
        construction = "ACETLSM"
    result = result.rename("aet")
    result.attrs.update(
        units="mm",
        long_name="CONUS404 accumulated net actual evapotranspiration",
        construction=construction,
    )
    return result

def total_runoff(dataset: xr.Dataset) -> xr.DataArray:
    result = (dataset["ACRUNSF"] + dataset["ACQRF"]).rename("runoff")
    result.attrs.update(
        units="mm",
        long_name="CONUS404 accumulated total runoff",
        construction="ACRUNSF + ACQRF",
    )
    return result

native_p = native_precipitation(native_ds) if EXPORT_NATIVE_PRECIPITATION else None
native_aet = actual_evapotranspiration(native_ds) if EXPORT_AET else None
native_runoff = total_runoff(native_ds) if EXPORT_RUNOFF else None
bias_adjusted_p = (
    bias_adjusted_precipitation(bias_adjusted_ds)
    if EXPORT_BIAS_ADJUSTED_PRECIPITATION
    else None
)

print("Native precipitation:", None if native_p is None else native_p.attrs["construction"])
print("AET:", None if native_aet is None else native_aet.attrs["construction"])
print("Runoff:", None if native_runoff is None else native_runoff.attrs["construction"])
print("Bias-adjusted precipitation:", None if bias_adjusted_p is None else "RAIN")


## 5. Cloud-access preview

The workflow reads a sparse one-day sample to record cloud access, variable names, and basic value ranges without launching the annual or seasonal export.


In [ ]:
def preview_field(field: xr.DataArray, label: str, date: str = "1981-01-15") -> dict:
    sample = (
        field.sel(time=date, method="nearest")
        .isel(x=slice(None, None, 100), y=slice(None, None, 100))
        .compute()
    )
    values = np.asarray(sample.values, dtype="float64")
    return {
        "field": label,
        "date": str(pd.Timestamp(sample.time.values).date()),
        "minimum_mm": float(np.nanmin(values)),
        "mean_mm": float(np.nanmean(values)),
        "maximum_mm": float(np.nanmax(values)),
        "sample_cells": int(np.isfinite(values).sum()),
    }

preview_rows = []
if RUN_PREVIEW:
    if native_p is not None:
        preview_rows.append(preview_field(native_p, "native precipitation"))
    if native_aet is not None:
        preview_rows.append(preview_field(native_aet, "AET"))
    if native_runoff is not None:
        preview_rows.append(preview_field(native_runoff, "runoff"))
    if bias_adjusted_p is not None:
        preview_rows.append(preview_field(bias_adjusted_p, "bias-adjusted precipitation"))
display(pd.DataFrame(preview_rows))


## 6. AET-definition diagnostic

The workflow calculates `(ACEDIR + ACETRAN + ACECAN) - ACETLSM` diagnostics for a sparse one-day sample. The workflow reports the mean difference, mean absolute difference, and maximum absolute difference without changing `AET_METHOD`.


In [ ]:
aet_diagnostic = None
aet_variables = {"ACEDIR", "ACETRAN", "ACECAN", "ACETLSM"}
if RUN_PREVIEW and aet_variables.issubset(native_ds.data_vars):
    component_sum = native_ds["ACEDIR"] + native_ds["ACETRAN"] + native_ds["ACECAN"]
    diagnostic_difference = (
        (component_sum - native_ds["ACETLSM"])
        .sel(time="1981-01-15", method="nearest")
        .isel(x=slice(None, None, 100), y=slice(None, None, 100))
        .compute()
    )
    difference_values = np.asarray(diagnostic_difference.values, dtype="float64")
    aet_diagnostic = {
        "mean_component_minus_ACETLSM_mm": float(np.nanmean(difference_values)),
        "mean_absolute_difference_mm": float(np.nanmean(np.abs(difference_values))),
        "maximum_absolute_difference_mm": float(np.nanmax(np.abs(difference_values))),
    }
aet_diagnostic


## 7. Aggregation and GeoTIFF functions

The workflow sums each period independently and releases it before processing the next period. This avoids constructing a full 40-year in-memory array. The workflow retains missing output cells when every timestep is missing by using `min_count=1`.


In [ ]:
SEASONS = {
    "Fall": {"start": (10, 1), "end": (12, 31), "suffix": "1230"},
    "Winter": {"start": (1, 1), "end": (3, 31), "suffix": "0330"},
    "Spring": {"start": (4, 1), "end": (6, 30), "suffix": "0630"},
    "Summer": {"start": (7, 1), "end": (9, 30), "suffix": "0930"},
}

def aggregate_sum(field: xr.DataArray, start: str, end: str, label: str) -> xr.DataArray:
    subset = field.sel(time=slice(start, end))
    count = subset.sizes.get("time", 0)
    if count == 0:
        raise ValueError(f"No timesteps found for {label}: {start} through {end}")
    result = subset.sum("time", skipna=True, min_count=1, keep_attrs=True)
    result.attrs.update(
        aggregation="sum",
        period_start=start,
        period_end=end,
        source_timestep_count=count,
    )
    return result

def source_crs(dataset: xr.Dataset):
    if "crs" not in dataset:
        raise KeyError("The source dataset does not contain the expected 'crs' coordinate.")
    return dataset["crs"].metpy.cartopy_crs

def prepare_raster(
    aggregate: xr.DataArray,
    source_dataset: xr.Dataset,
    target_crs: str = TARGET_CRS,
) -> xr.DataArray:
    raster = aggregate.squeeze(drop=True)
    if not {"x", "y"}.issubset(raster.dims):
        raise ValueError(f"Expected x/y dimensions; found {raster.dims}")
    raster = raster.rio.set_spatial_dims(x_dim="x", y_dim="y")
    raster = raster.rio.write_crs(source_crs(source_dataset))
    removable_coordinates = [name for name in ("lat", "lon") if name in raster.coords]
    if removable_coordinates:
        raster = raster.drop_vars(removable_coordinates)
    if raster.rio.crs != target_crs:
        raster = raster.rio.reproject(target_crs, nodata=OUTPUT_NODATA)
    raster = raster.astype("float32")
    raster.attrs.update(aggregate.attrs)
    return raster

def write_geotiff(
    aggregate: xr.DataArray,
    source_dataset: xr.Dataset,
    destination: Path,
) -> str:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and not OVERWRITE:
        return f"SKIP existing: {destination.relative_to(REPO_ROOT)}"
    raster = prepare_raster(aggregate, source_dataset)
    raster.rio.to_raster(
        destination,
        driver="GTiff",
        dtype="float32",
        compress="DEFLATE",
        predictor=3,
        tiled=True,
        BIGTIFF="IF_SAFER",
    )
    return f"WROTE: {destination.relative_to(REPO_ROOT)}"

def annual_destination(product: str, water_year: int) -> Path:
    if product == "native_precipitation":
        return OUTPUT_DIRECTORIES["annual_native_precipitation"] / (
            f"RAIN_Mean_Annual_precipitation_WY{water_year}.tif"
        )
    if product == "bias_adjusted_precipitation":
        return OUTPUT_DIRECTORIES["annual_bias_adjusted_precipitation"] / (
            f"RAIN_Mean_Annual_precipitation_WY{water_year}.tif"
        )
    if product == "aet":
        return OUTPUT_DIRECTORIES["annual_aet"] / f"CONUS_AET_WY{water_year}.tif"
    if product == "runoff":
        return OUTPUT_DIRECTORIES["annual_runoff"] / f"CONUS404_Runoff_WY{water_year}.tif"
    raise KeyError(product)

def seasonal_destination(product: str, season: str, year: int) -> Path:
    suffix = SEASONS[season]["suffix"]
    if product == "native_precipitation":
        return OUTPUT_DIRECTORIES["seasonal_native_precipitation"] / (
            f"ET_Mean_Seasonal_RAIN_CONUS404_{year}{suffix}.tif"
        )
    if product == "bias_adjusted_precipitation":
        return OUTPUT_DIRECTORIES["seasonal_bias_adjusted_precipitation"] / (
            f"ET_Mean_Seasonal_RAIN_CONUS404_{year}{suffix}.tif"
        )
    if product == "aet":
        return OUTPUT_DIRECTORIES["seasonal_aet"] / (
            f"CONUS404_AET_{season}_{year}{suffix}.tif"
        )
    raise KeyError(product)


## 8. Export manifest

The workflow constructs a manifest containing every requested product, aggregation interval, label, and destination. The workflow labels annual dates by water year and seasonal dates by calendar year.


In [ ]:
def selected_products() -> list[tuple[str, xr.DataArray, xr.Dataset]]:
    products = []
    if native_p is not None:
        products.append(("native_precipitation", native_p, native_ds))
    if bias_adjusted_p is not None:
        products.append(
            ("bias_adjusted_precipitation", bias_adjusted_p, bias_adjusted_ds)
        )
    if native_aet is not None:
        products.append(("aet", native_aet, native_ds))
    if native_runoff is not None:
        products.append(("runoff", native_runoff, native_ds))
    return products

manifest_rows = []
products = selected_products()

if EXPORT_ANNUAL:
    for water_year in range(START_WATER_YEAR, END_WATER_YEAR + 1):
        start = f"{water_year - 1}-10-01"
        end = f"{water_year}-09-30"
        for product, _, _ in products:
            manifest_rows.append(
                {
                    "scale": "annual",
                    "product": product,
                    "label": f"WY{water_year}",
                    "start": start,
                    "end": end,
                    "destination": str(annual_destination(product, water_year).relative_to(REPO_ROOT)),
                }
            )

if EXPORT_SEASONAL:
    for year in range(START_SEASON_YEAR, END_SEASON_YEAR + 1):
        for season, definition in SEASONS.items():
            start_month, start_day = definition["start"]
            end_month, end_day = definition["end"]
            start = f"{year}-{start_month:02d}-{start_day:02d}"
            end = f"{year}-{end_month:02d}-{end_day:02d}"
            for product, _, _ in products:
                if product == "runoff":
                    continue
                manifest_rows.append(
                    {
                        "scale": "seasonal",
                        "product": product,
                        "label": f"{season} {year}",
                        "start": start,
                        "end": end,
                        "destination": str(
                            seasonal_destination(product, season, year).relative_to(REPO_ROOT)
                        ),
                    }
                )

manifest = pd.DataFrame(manifest_rows)
print(f"Planned files: {len(manifest):,}")
display(manifest.head(12))
display(manifest.groupby(["scale", "product"]).size().rename("file_count"))


## 9. GeoTIFF export

The workflow computes all selected variables for one period together, writes compressed float32 GeoTIFFs in EPSG:5070, and then advances to the next period. It retains existing outputs unless `OVERWRITE = True`.


In [ ]:
def export_manifest(manifest: pd.DataFrame) -> pd.DataFrame:
    log_rows = []
    product_lookup = {
        product: (field, dataset) for product, field, dataset in selected_products()
    }

    for (scale, label), period_rows in manifest.groupby(["scale", "label"], sort=False):
        print(f"Processing {scale}: {label}")
        pending = []
        for row in period_rows.itertuples(index=False):
            destination = REPO_ROOT / row.destination
            if destination.exists() and not OVERWRITE:
                log_rows.append(
                    {
                        "scale": scale,
                        "label": label,
                        "product": row.product,
                        "status": "skipped",
                        "destination": row.destination,
                    }
                )
                continue
            field, source_dataset = product_lookup[row.product]
            aggregate = aggregate_sum(field, row.start, row.end, f"{label} {row.product}")
            pending.append((row, aggregate, source_dataset, destination))

        if not pending:
            continue

        computed = dask.compute(*(item[1] for item in pending))
        for (row, _, source_dataset, destination), aggregate in zip(pending, computed):
            message = write_geotiff(aggregate, source_dataset, destination)
            print(" ", message)
            log_rows.append(
                {
                    "scale": scale,
                    "label": label,
                    "product": row.product,
                    "status": "written",
                    "destination": row.destination,
                }
            )
    return pd.DataFrame(log_rows)

if RUN_EXPORTS:
    export_log = export_manifest(manifest)
    display(export_log)
else:
    export_log = pd.DataFrame()
    print("RUN_EXPORTS is False; no GeoTIFFs were written.")


## 10. Output summary

The workflow summarizes the first file written in the run by its shape, CRS, resolution, nodata value, finite-value statistics, and map.


In [ ]:
if not export_log.empty and (export_log["status"] == "written").any():
    first_relative_path = export_log.loc[
        export_log["status"] == "written", "destination"
    ].iloc[0]
    first_path = REPO_ROOT / first_relative_path
    with rioxarray.open_rasterio(first_path, masked=True) as check:
        values = np.asarray(check.values, dtype="float64")
        qc = {
            "path": first_relative_path,
            "shape": check.shape,
            "crs": str(check.rio.crs),
            "resolution": check.rio.resolution(),
            "nodata": check.rio.nodata,
            "minimum": float(np.nanmin(values)),
            "mean": float(np.nanmean(values)),
            "maximum": float(np.nanmax(values)),
            "finite_cells": int(np.isfinite(values).sum()),
        }
    display(qc)

    with rioxarray.open_rasterio(first_path, masked=True) as check:
        check.squeeze(drop=True).plot(
            robust=True,
            cmap="viridis",
            figsize=(10, 6),
        )
        plt.title(first_path.name)
        plt.show()
else:
    print("No new output is available for QC in this run.")


## 11. Close Dask client

The workflow closes the local Dask client after processing. The full CONUS404 record began in October 1979 and ended in September 2022; the complete four-season calendar-year range was 1980–2021.


In [ ]:
if client is not None:
    client.close()
    print("Dask client closed.")
